# Домашнее задание №2 — сегментация YOLOv8-seg
**Литвинец И.Д., ИУ5-21М**

Тема сохраняется из ДЗ-1: **заяц (hare), кролик (rabbit), пищуха (pika)**.

Порядок классов ВЕЗДЕ одинаковый — в CVAT, в data.yaml и в labels.json React-приложения:
```
0 — hare   (Заяц)
1 — rabbit (Кролик)
2 — pika   (Пищуха)
```

Задание требует обучить YOLOv8-segment **не менее 4 раз**. Ниже — 4 готовые конфигурации,
как в лабораторных: меняете `EXP`, запускаете ячейки заново.

Работает в **Google Colab** (нужен GPU для разумной скорости; сегментация тяжелее
обычной детекции). Локально на своём ПК тоже можно — Ultralytics ставится и на Windows
с CUDA, но Colab избавляет от возни с драйверами ради разовой задачи.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
# Если не хотите использовать Google Drive — закомментируйте строку выше
# и в следующей ячейке поставьте DATASET_ROOT = '/content/dataset'
# (тогда всё сотрётся при отключении среды выполнения — качайте датасет заново).

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q ultralytics onnx onnx-simplifier onnxruntime scikit-learn
import ultralytics
ultralytics.checks()

## 1. Загрузка размеченного датасета

Из CVAT экспортируйте задачу в формате **COCO 1.0** (Menu → Export task dataset →
COCO 1.0). Если используете бесплатный app.cvat.ai — картинки в архив не попадают,
нужно отдельно скачать те же изображения и положить рядом с `instances_default.json`
в одну папку.

Загрузите на Google Drive папку вида:
```
MyDrive/dz2_dataset/
    instances_default.json
    hare_0001.jpg
    hare_0002.jpg
    rabbit_0001.jpg
    ...
```

In [ ]:
DATASET_ROOT = '/content/gdrive/MyDrive/dz2_dataset'   # <-- поправьте путь под себя

import os
print('Файлов в папке:', len(os.listdir(DATASET_ROOT)))
print('instances_default.json на месте:',
     os.path.exists(os.path.join(DATASET_ROOT, 'instances_default.json')))

## 2. Конвертация разметки CVAT (COCO) в формат YOLO-segmentation

CVAT/COCO хранит полигоны как список координат `[x1,y1,x2,y2,...]` в пикселях.
YOLO-segmentation хочет то же самое, но **нормализованное** (0…1) и в отдельном
`.txt`-файле на каждую картинку — по одной строке на объект:
```
класс  x1 y1  x2 y2  x3 y3  ...
```

In [ ]:
import json as _json

def convert_coco_to_yolo_segmentation(json_file, out_dir='labels'):
    """Переписывает полигоны из COCO JSON в формат YOLO-segmentation."""
    with open(json_file, 'r', encoding='utf-8') as f:
        coco = _json.load(f)

    out_path = os.path.join(os.path.dirname(json_file), out_dir)
    os.makedirs(out_path, exist_ok=True)

    images_by_id = {im['id']: im for im in coco['images']}
    # CVAT нумерует категории с 1 — сдвигаем к 0, чтобы совпадало с data.yaml
    cat_id_to_idx = {c['id']: i for i, c in enumerate(coco['categories'])}
    print('Классы в разметке (в порядке category_id):',
         [c['name'] for c in coco['categories']])

    written = 0
    for ann in coco['annotations']:
        img = images_by_id[ann['image_id']]
        w, h = img['width'], img['height']
        stem = os.path.splitext(os.path.basename(img['file_name']))[0]
        cls_idx = cat_id_to_idx[ann['category_id']]

        poly = ann['segmentation'][0]           # берём первый полигон объекта
        norm = [f'{poly[i] / (w if i % 2 == 0 else h):.6f}'
               for i in range(len(poly))]
        line = f"{cls_idx} {' '.join(norm)}\n"

        with open(os.path.join(out_path, stem + '.txt'), 'a', encoding='utf-8') as f:
            f.write(line)
        written += 1

    print(f'Готово. Строк разметки записано: {written}')
    return [c['name'] for c in coco['categories']]

json_path = os.path.join(DATASET_ROOT, 'instances_default.json')
class_names_in_cvat = convert_coco_to_yolo_segmentation(json_path)

## 3. Разбиение на train / val / test

Правило то же, что во всех лабораторных: разбиение только по изображениям,
никогда не смешиваем один и тот же объект между выборками (здесь это неактуально —
каждый объект уже в одном изображении).

In [ ]:
import shutil
from sklearn.model_selection import train_test_split

TEST_SIZE, VAL_SIZE = 0.15, 0.15
SEED = 42

images = [f for f in os.listdir(DATASET_ROOT)
         if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
labels_dir = os.path.join(DATASET_ROOT, 'labels')
images_with_labels = [f for f in images
                      if os.path.exists(os.path.join(
                          labels_dir, os.path.splitext(f)[0] + '.txt'))]

print(f'Изображений всего: {len(images)}, с разметкой: {len(images_with_labels)}')
if len(images) != len(images_with_labels):
    print('ВНИМАНИЕ: есть изображения без разметки — они не попадут в датасет.')

train_files, test_files = train_test_split(
    images_with_labels, test_size=TEST_SIZE, random_state=SEED)
train_files, val_files = train_test_split(
    train_files, test_size=VAL_SIZE / (1 - TEST_SIZE), random_state=SEED)

def place(files, split):
    for f in files:
        for sub, ext_src, ext_dst in [('images', f, f),
                                      ('labels', os.path.splitext(f)[0] + '.txt',
                                       os.path.splitext(f)[0] + '.txt')]:
            dst_dir = os.path.join(DATASET_ROOT, sub, split)
            os.makedirs(dst_dir, exist_ok=True)
            src = os.path.join(DATASET_ROOT, ext_src) if sub == 'images' \
                 else os.path.join(labels_dir, ext_src)
            shutil.copy2(src, os.path.join(dst_dir, ext_dst))

for split, files in [('train', train_files), ('val', val_files), ('test', test_files)]:
    place(files, split)
    print(f'{split:5s}: {len(files)} изображений')

In [ ]:
import yaml

CLASS_NAMES = {0: 'hare', 1: 'rabbit', 2: 'pika'}   # тот же порядок, что в CVAT-проекте

data_yaml = {
    'path':  DATASET_ROOT,
    'train': os.path.join(DATASET_ROOT, 'images', 'train'),
    'val':   os.path.join(DATASET_ROOT, 'images', 'val'),
    'test':  os.path.join(DATASET_ROOT, 'images', 'test'),
    'names': CLASS_NAMES,
}
yaml_path = os.path.join(DATASET_ROOT, 'data.yaml')
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data_yaml, f, allow_unicode=True)

print(open(yaml_path, encoding='utf-8').read())

## 4. Журнал результатов

Как и в лабораторных — числа автоматически копятся в JSON, никакая конфигурация
не потеряется.

In [ ]:
RESULTS_PATH = '/content/results_dz2.json'

def load_results():
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH, 'r', encoding='utf-8') as f:
            return _json.load(f)
    return []

def log_result(**kw):
    res = [r for r in load_results() if r.get('exp') != kw.get('exp')]
    res.append(kw)
    with open(RESULTS_PATH, 'w', encoding='utf-8') as f:
        _json.dump(res, f, ensure_ascii=False, indent=2)
    print('Записано в журнал:', kw.get('exp'))

def show_results():
    import pandas as pd
    res = load_results()
    if not res:
        print('Журнал пуст'); return None
    df = pd.DataFrame(res)
    display(df)
    return df

## 5. Конфигурация экспериментов — задание требует не менее 4 прогонов

Раскомментируйте одну строку `EXP` и выполните ячейки обучения ниже. Логика подбора:

* **e1_base** — модель `yolov8n-seg` (самая лёгкая), стандартные настройки
* **e2_epochs** — вдвое больше эпох: проверяем, не мешает ли модели короткое обучение
* **e3_small** — модель `yolov8s-seg` (больше, чем nano): ёмкость важнее числа эпох?
* **e4_imgsz** — увеличенное разрешение входа: детали мельких животных видны лучше

In [ ]:
EXPERIMENTS = {
    'e1_base':   dict(model='yolov8n-seg.pt', epochs=40,  imgsz=640, batch=16,
                      note='Базовая: yolov8n-seg, 40 эпох, 640px'),
    'e2_epochs': dict(model='yolov8n-seg.pt', epochs=80,  imgsz=640, batch=16,
                      note='Эпох x2: проверяем недообучение'),
    'e3_small':  dict(model='yolov8s-seg.pt', epochs=40,  imgsz=640, batch=8,
                      note='Модель s вместо n: больше ёмкость'),
    'e4_imgsz':  dict(model='yolov8n-seg.pt', epochs=40,  imgsz=960, batch=8,
                      note='Разрешение 960 вместо 640'),
}

# ================== ВЫБЕРИТЕ ЭКСПЕРИМЕНТ ==================
EXP = 'e1_base'
# EXP = 'e2_epochs'
# EXP = 'e3_small'
# EXP = 'e4_imgsz'
# ==========================================================

CFG = EXPERIMENTS[EXP]
print(f'Эксперимент: {EXP}')
print(f'  {CFG}')

## 6. Обучение

`yolov8*-seg.pt` — предобученные на COCO веса, поэтому это тоже перенос обучения:
сеть уже умеет выделять границы объектов вообще, дообучение учит её узнавать именно
ваши три класса.

In [ ]:
import time
from ultralytics import YOLO

model = YOLO(CFG['model'])

t0 = time.time()
train_results = model.train(
    data=yaml_path,
    epochs=CFG['epochs'],
    imgsz=CFG['imgsz'],
    batch=CFG['batch'],
    device=0,
    project='/content/gdrive/MyDrive/dz2_runs',
    name=EXP,
    exist_ok=True,
)
train_time = time.time() - t0
print(f'\nОбучение заняло {train_time:.1f} с')

## 7. Оценка на тестовой выборке

In [ ]:
val_results = model.val(data=yaml_path, split='test')
rd = val_results.results_dict
print(rd)

# основные метрики: box — рамки, mask (M) — сама сегментация (что важнее для этой лабы)
box_map50   = rd.get('metrics/mAP50(B)')
box_map     = rd.get('metrics/mAP50-95(B)')
mask_map50  = rd.get('metrics/mAP50(M)')
mask_map    = rd.get('metrics/mAP50-95(M)')
precision_m = rd.get('metrics/precision(M)')
recall_m    = rd.get('metrics/recall(M)')

print(f'\nМаска mAP50: {mask_map50:.4f}   mAP50-95: {mask_map:.4f}')
print(f'Маска precision: {precision_m:.4f}   recall: {recall_m:.4f}')

In [ ]:
weights_path = f'/content/gdrive/MyDrive/dz2_runs/{EXP}/weights/best.pt'

log_result(
    exp=EXP,
    model=CFG['model'], epochs=CFG['epochs'], imgsz=CFG['imgsz'], batch=CFG['batch'],
    box_map50=round(float(box_map50), 4),
    box_map=round(float(box_map), 4),
    mask_map50=round(float(mask_map50), 4),
    mask_map=round(float(mask_map), 4),
    mask_precision=round(float(precision_m), 4),
    mask_recall=round(float(recall_m), 4),
    train_time_s=round(train_time, 1),
    note=CFG['note'],
)

## 8. Пример предсказания — картинка для отчёта

In [ ]:
import glob
test_img = glob.glob(os.path.join(DATASET_ROOT, 'images', 'test', '*'))[0]
pred = model(test_img)
pred[0].save(filename=f'/content/gdrive/MyDrive/dz2_runs/{EXP}_predict_example.jpg')
pred[0].show()

## 9. Экспорт в ONNX

Экспортируется **лучшая** конфигурация — определите её после того, как прогоните
все 4 эксперимента и сравните `mask_map50` в итоговой таблице (раздел 11).
Перезапустите эту ячейку с `EXP`, выставленным на победителя.

In [ ]:
best_model = YOLO(f'/content/gdrive/MyDrive/dz2_runs/{EXP}/weights/best.pt')
onnx_path = best_model.export(format='onnx', imgsz=CFG['imgsz'], simplify=True)
print('Экспортировано:', onnx_path)
print('Скопируйте этот файл как model.onnx в React-проект: public/model/model.onnx')

## 10. Проверка ONNX-модели

In [ ]:
onnx_model = YOLO(onnx_path)
result = onnx_model(test_img)
print('ONNX-модель отработала, объектов найдено:', len(result[0].boxes))

## 11. Итоговая таблица по всем 4 экспериментам

In [ ]:
df = show_results()
if df is not None:
    df.to_csv('/content/gdrive/MyDrive/dz2_runs/results_dz2.csv',
              index=False, encoding='utf-8-sig')
    print('Сохранено: results_dz2.csv')